# Step 7: Model Selection and Baseline Modeling

## Purpose
This notebook builds simple baseline models for the Telco churn project and compares them fairly.

## Why this step matters
Before tuning or using more advanced models, we need to understand what level of performance is easy to achieve. A good baseline prevents us from overestimating the value of a more complex model.

## Main goal
Build a small set of baseline models, compare them on the validation set, and identify the most promising model families for the next step.

In [18]:
from pathlib import Path

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

## 1. Rebuild the feature set and the split

We reuse the same cleaned data, engineered features, and train/validation/test split logic from earlier steps.

## Why?
Because baseline model comparison should be fair. If different models see different data setups, then the comparison becomes unreliable.

In [19]:
PROJECT_DIR = Path.cwd()
CLEANED_FILE = PROJECT_DIR / "WA_Fn-UseC_-Telco-Customer-Churn-cleaned.csv"

df = pd.read_csv(CLEANED_FILE)

df['service_count'] = (
    (df['PhoneService'] == 'Yes').astype(int)
    + (df['MultipleLines'] == 'Yes').astype(int)
    + (df['InternetService'] != 'No').astype(int)
    + (df['OnlineSecurity'] == 'Yes').astype(int)
    + (df['OnlineBackup'] == 'Yes').astype(int)
    + (df['DeviceProtection'] == 'Yes').astype(int)
    + (df['TechSupport'] == 'Yes').astype(int)
    + (df['StreamingTV'] == 'Yes').astype(int)
    + (df['StreamingMovies'] == 'Yes').astype(int)
)

df['avg_monthly_value_from_total'] = (df['TotalCharges'] / df['tenure'].replace(0, 1)).round(2)
df['is_new_customer'] = (df['tenure'] <= 12).astype(int)
df['has_long_term_contract'] = df['Contract'].isin(['One year', 'Two year']).astype(int)

X = df.drop(columns=['customerID', 'Churn']).copy()
y = df['Churn'].map({'No': 0, 'Yes': 1}).copy()

categorical_features = X.select_dtypes(include='object').columns.tolist()
numeric_features = X.select_dtypes(exclude='object').columns.tolist()

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_full,
    y_train_full,
    test_size=0.25,
    random_state=42,
    stratify=y_train_full,
)

X_train.shape, X_val.shape, X_test.shape

((4218, 23), (1407, 23), (1407, 23))

## 2. Define a shared preprocessing pipeline

We use the same preprocessing for all baseline models.

## Why?
If one model gets scaled and encoded inputs while another does not, the comparison becomes unfair. A shared preprocessing step helps us compare models based on model behavior, not preprocessing differences.

In [20]:
preprocessor = ColumnTransformer(
    transformers=[
        ('numeric', StandardScaler(), numeric_features),
        ('categorical', OneHotEncoder(handle_unknown='ignore'), categorical_features),
    ]
)

## 3. Choose baseline models

We begin with a small but informative set of models.

- **DummyClassifier**: tells us how a trivial baseline performs
- **LogisticRegression**: strong interpretable baseline for classification
- **DecisionTreeClassifier**: simple non-linear model
- **RandomForestClassifier**: stronger ensemble tree baseline

## Why these models?
Together they show us whether the problem is mostly linear, whether simple non-linear rules help, and whether a stronger tree ensemble gives a meaningful lift.

## 4. Define an evaluation function

We evaluate on the **validation set**, not the test set.

## Why?
Because Step 7 is still a decision-making phase. We are comparing model families, so the test set must remain untouched.

We track multiple metrics:
- `accuracy`
- `precision`
- `recall`
- `f1`
- `roc_auc`

We care especially about `recall`, because your stated goal is to catch churners. But we still keep other metrics so we do not ignore false positives or weak ranking behavior.

In [21]:
models = {
    'Dummy Most Frequent': DummyClassifier(strategy='most_frequent'),
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42, max_depth=5),
    'Random Forest': RandomForestClassifier(random_state=42, n_estimators=200, max_depth=8),
}

In [22]:
def evaluate_model(model_name, model, X_train, y_train, X_val, y_val):
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('model', model),
    ])

    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_val)

    if hasattr(pipeline, 'predict_proba'):
        y_scores = pipeline.predict_proba(X_val)[:, 1]
    else:
        y_scores = y_pred

    return {
        'model': model_name,
        'accuracy': round(accuracy_score(y_val, y_pred), 4),
        'precision': round(precision_score(y_val, y_pred, zero_division=0), 4),
        'recall': round(recall_score(y_val, y_pred, zero_division=0), 4),
        'f1': round(f1_score(y_val, y_pred, zero_division=0), 4),
        'roc_auc': round(roc_auc_score(y_val, y_scores), 4),
    }

In [23]:
results = []

for model_name, model in models.items():
    results.append(evaluate_model(model_name, model, X_train, y_train, X_val, y_val))

results_df = pd.DataFrame(results).sort_values(by=['recall', 'f1', 'roc_auc'], ascending=False)
results_df

,model,accuracy,precision,recall,f1,roc_auc
1,Logistic Regression,0.8010,0.6643,0.5080,0.5758,0.8345
3,Random Forest,0.7946,0.6641,0.4599,0.5434,0.8350
2,Decision Tree,0.7804,0.6255,0.4332,0.5118,0.8148
0,Dummy Most Frequent,0.7342,0.0000,0.0000,0.0000,0.5000


In [24]:
results_df

,model,accuracy,precision,recall,f1,roc_auc
1,Logistic Regression,0.8010,0.6643,0.5080,0.5758,0.8345
3,Random Forest,0.7946,0.6641,0.4599,0.5434,0.8350
2,Decision Tree,0.7804,0.6255,0.4332,0.5118,0.8148
0,Dummy Most Frequent,0.7342,0.0000,0.0000,0.0000,0.5000


## 5. How to interpret the baseline results

When you run the results table, read it like this:

- **Dummy model** tells us the minimum benchmark. If a real model barely beats it, that is a warning sign.
- **Logistic Regression** tells us how well a simple, interpretable linear model works.
- **Decision Tree** tells us whether simple non-linear decision rules help.
- **Random Forest** tells us whether a stronger tree-based ensemble gives a meaningful lift.

## What to look for
- If **Logistic Regression** performs strongly, that is good news for interpretation.
- If **Random Forest** is much better, then non-linear relationships matter.
- If **Decision Tree** performs poorly but **Random Forest** performs well, then a single tree may be too simple while an ensemble captures more stable patterns.
- If every real model only slightly beats the dummy baseline, then either the signal is weak, the features need improvement, or the evaluation setup needs re-checking.

## Shortlist rule for the next step
For Step 8, we usually keep:
- one strong interpretable model
- one strong non-linear model

That gives us a balanced shortlist for tuning and deeper evaluation.

In [25]:
# Duplicate helper cell intentionally neutralized during cleanup.
# Use the earlier `evaluate_model` definition above.
"Step 7 cleanup complete."

'Step 7 cleanup complete.'